# Introduction to Differential Expression Analysis

In this tutorial you will perform a complete bulk RNA-seq differential expression (DE)
analysis using **PyDESeq2** — a Python implementation of the widely-used DESeq2 method.

## What you will learn

1. Load a raw count matrix and sample metadata.
2. Filter low-count genes.
3. Run the DESeq2 statistical model with PyDESeq2.
4. Extract and interpret DE results.
5. Visualize results with MA plots, volcano plots, and heatmaps.
6. Export results.
7. Run the nf-core `differentialabundance` pipeline and compare its output to your manual analysis.

## Resources

- 📖 **PyDESeq2 documentation**: https://pydeseq2.readthedocs.io/
- 📖 **DESeq2 paper** (Love *et al.*, 2014): https://doi.org/10.1186/s13059-014-0550-8
- 📖 **DESeq2 vignette (R)**: https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html
- 📖 **nf-core differentialabundance**: https://nf-co.re/differentialabundance

## Data

The input data is stored in the `data/` folder:

| File | Description |
|------|-------------|
| `data/counts.tsv` | Raw gene-level count matrix (genes × samples) |
| `data/metadata.tsv` | Sample metadata (sample IDs, condition labels, …) |

> **Note:** Do **not** commit large data files. They are excluded by `.gitignore`.

---

## 0 — Imports

Import all libraries needed for this analysis.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

warnings.filterwarnings("ignore")

# ── reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── output folder ────────────────────────────────────────────────────────────
import pathlib
OUTDIR = pathlib.Path("output")
OUTDIR.mkdir(exist_ok=True)

print("All imports successful.")

---
## 1 — Load Data

Load the raw count matrix and sample metadata from the `data/` folder.

### Count matrix format

The count matrix should be a tab-separated file where:
- **Rows** are genes (identified by a gene ID or gene symbol).
- **Columns** are samples.
- Values are **integer raw counts** (not normalized, not log-transformed).

### Metadata format

The metadata file should be tab-separated with:
- One row per sample.
- A column matching the sample names in the count matrix.
- A `condition` column (or similar) indicating the group each sample belongs to (e.g. `"treated"` vs `"control"`).

> **Hint:** Use `pd.read_csv(..., sep="\t", index_col=0)` to load TSV files with a row-index column.

In [ ]:
# Load raw count matrix
counts = pd.read_csv("data/counts.tsv", sep="\t", index_col=0)

# Load sample metadata
metadata = pd.read_csv("data/metadata.tsv", sep="\t", index_col=0)

print(f"Count matrix shape: {counts.shape}  (genes × samples)")
print(f"Metadata shape:     {metadata.shape}  (samples × variables)")
print()
print("First few rows of the count matrix:")
display(counts.head())
print()
print("Metadata:")
display(metadata)

In [ ]:
# Verify that the sample names are consistent between the two files.
# The count matrix columns must match the metadata index.
assert set(counts.columns) == set(metadata.index), (
    "Sample names in counts.columns and metadata.index do not match!\n"
    f"  counts columns:   {sorted(counts.columns.tolist())}\n"
    f"  metadata samples: {sorted(metadata.index.tolist())}"
)

# Align ordering
counts = counts[metadata.index]

print("Sample names are consistent. ✔")
print()
print("Conditions in metadata:")
print(metadata["condition"].value_counts())

---
## 2 — Pre-processing: Filtering Low-Count Genes

Genes with very low counts across all samples carry little statistical power and add
noise. A standard approach is to keep only genes with a minimum total count (or
a minimum number of samples with at least one count).

### Common filtering strategies

| Strategy | Description |
|----------|-------------|
| Minimum total count | Keep genes whose **sum across all samples** ≥ threshold |
| Minimum count per sample | Keep genes with at least *k* counts in at least *n* samples |

PyDESeq2 applies an independent filtering step internally as well, but pre-filtering
reduces computation time and helps model convergence.

> **Hint:** A commonly used threshold is to keep genes with at least **10 counts** in at least **2 samples**.

In [ ]:
MIN_COUNTS = 10   # minimum count value
MIN_SAMPLES = 2   # minimum number of samples that must exceed MIN_COUNTS

n_before = counts.shape[0]

keep = (counts >= MIN_COUNTS).sum(axis=1) >= MIN_SAMPLES
counts_filtered = counts.loc[keep]

n_after = counts_filtered.shape[0]
print(f"Genes before filtering: {n_before:,}")
print(f"Genes after  filtering: {n_after:,}")
print(f"Genes removed:          {n_before - n_after:,}")

In [ ]:
# Visualise the count distribution (log10 total counts per gene)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, title) in zip(
    axes,
    [(counts, "Before filtering"), (counts_filtered, "After filtering")]
):
    total = df.sum(axis=1)
    ax.hist(np.log10(total + 1), bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_xlabel("log10(total counts + 1)")
    ax.set_ylabel("Number of genes")
    ax.set_title(title)

fig.suptitle("Gene count distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTDIR / "count_distribution.png", dpi=150)
plt.show()

---
## 3 — Run PyDESeq2

PyDESeq2 implements the **DESeq2** statistical framework in pure Python.
The workflow has two main objects:

| Object | Purpose |
|--------|---------|
| `DeseqDataSet` | Stores counts + metadata; estimates size factors and dispersions |
| `DeseqStats` | Performs Wald tests and corrects for multiple comparisons |

### Key concepts

- **Size factors** — account for differences in sequencing depth between samples.
- **Dispersion** — models the gene-level variance (biological + technical noise).
- **Wald test** — tests whether the log2 fold change for a given contrast is significantly different from zero.
- **Benjamini–Hochberg (BH) correction** — controls the false discovery rate (FDR).

> **Resources:**
> - https://pydeseq2.readthedocs.io/en/latest/auto_examples/plot_pandas_DESeq2.html
> - https://pydeseq2.readthedocs.io/en/latest/api.html

### 3.1 — Build the `DeseqDataSet`

PyDESeq2 expects the count matrix in **samples × genes** orientation (transposed relative
to the standard bioinformatics convention of genes × samples).

In [ ]:
# PyDESeq2 expects counts as samples × genes (rows = samples)
counts_T = counts_filtered.T  # shape: samples × genes

# Make sure the counts are integer type
counts_T = counts_T.astype(int)

inference = DefaultInference(n_cpus=4)

dds = DeseqDataSet(
    counts=counts_T,
    metadata=metadata,
    design_factors="condition",   # column in metadata to test
    refit_cooks=True,
    inference=inference,
)

print(dds)

### 3.2 — Fit the Model

`dds.deseq2()` runs the full DESeq2 pipeline:
1. Estimates size factors (median-of-ratios method).
2. Estimates gene-wise dispersions.
3. Shrinks dispersions toward a fitted trend (empirical Bayes).
4. Fits negative binomial GLMs.
5. Refits for samples with high Cook's distance (outlier detection).

In [ ]:
dds.deseq2()
print("Model fitting complete. ✔")

### 3.3 — Dispersion Plot

Plotting the per-gene dispersion estimates against their mean counts is a standard
quality-check step. Expect to see:

- High dispersion for low-count genes.
- A downward trend as mean counts increase.
- Final (MAP) estimates shrunk toward the fitted trend (red curve).

In [ ]:
# Extract dispersion information from the fitted object
mean_counts = dds.layers["normed_counts"].mean(axis=0)   # mean normalised counts per gene
dispersions_gene = dds.varm["genewise_dispersions"]       # gene-wise MLEs
dispersions_map  = dds.varm["MAP_dispersions"]            # MAP (shrunk) estimates
trend            = dds.varm["fitted_dispersions"]         # parametric trend

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mean_counts, dispersions_gene, s=2, alpha=0.3, color="steelblue", label="Gene-wise MLE")
ax.scatter(mean_counts, dispersions_map,  s=2, alpha=0.5, color="black",     label="MAP estimate")

# Sort for a smooth trend line
order = np.argsort(mean_counts)
ax.plot(mean_counts[order], trend[order], color="red", lw=1.5, label="Fitted trend")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Mean of normalised counts")
ax.set_ylabel("Dispersion")
ax.set_title("Dispersion estimates")
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig(OUTDIR / "dispersion_plot.png", dpi=150)
plt.show()

### 3.4 — Run Wald Test and Extract Results

Define the **contrast** — which comparison to test.
A contrast is specified as `[factor, numerator_level, denominator_level]`.

For example, `["condition", "treated", "control"]` tests:

$$\log_2\!\left(\frac{\text{treated}}{\text{control}}\right)$$

Positive log2FC → higher expression in the numerator group (treated).  
Negative log2FC → lower expression in the numerator group (treated).

In [ ]:
# Adapt these values to match the levels in your metadata 'condition' column.
CONDITIONS = metadata["condition"].unique().tolist()
print("Available conditions:", CONDITIONS)

# Set numerator and denominator — change these if needed.
NUMERATOR   = CONDITIONS[0]
DENOMINATOR = CONDITIONS[1]

print(f"\nContrast: {NUMERATOR} vs {DENOMINATOR}  "
      f"(positive LFC = higher in '{NUMERATOR}')")

In [ ]:
stat_res = DeseqStats(
    dds,
    contrast=["condition", NUMERATOR, DENOMINATOR],
    inference=inference,
)

stat_res.summary()

results = stat_res.results_df
print(f"\nResult table shape: {results.shape}")
display(results.head(10))

In [ ]:
# Quick summary of significance
FDR_THRESHOLD = 0.05
LFC_THRESHOLD = 1.0   # |log2FC| > 1 corresponds to a 2-fold change

sig = results.dropna(subset=["padj"])
sig = sig[(sig["padj"] < FDR_THRESHOLD) & (sig["log2FoldChange"].abs() > LFC_THRESHOLD)]

print(f"Significant DE genes (FDR < {FDR_THRESHOLD}, |LFC| > {LFC_THRESHOLD}): {len(sig):,}")
print(f"  Up-regulated   (in {NUMERATOR}): {(sig['log2FoldChange'] > 0).sum():,}")
print(f"  Down-regulated (in {NUMERATOR}): {(sig['log2FoldChange'] < 0).sum():,}")

---
## 4 — Visualize Results

Three complementary visualizations are standard in DE analyses:

| Plot | X-axis | Y-axis | Purpose |
|------|--------|--------|---------|
| MA plot | Mean expression | log2FC | Assess fold-change vs expression level |
| Volcano plot | log2FC | −log10(p-value) | Highlight significant genes |
| Heatmap | Samples | Top DE genes | Show expression patterns across samples |

### 4.1 — MA Plot

An MA plot shows the **relationship between mean expression and fold change**.
Ideally, most genes should cluster around LFC = 0 (no change), with DE genes
scattered above/below.

In [ ]:
df_ma = results.dropna(subset=["padj", "log2FoldChange", "baseMean"]).copy()
df_ma["significant"] = (df_ma["padj"] < FDR_THRESHOLD) & (df_ma["log2FoldChange"].abs() > LFC_THRESHOLD)

fig, ax = plt.subplots(figsize=(8, 5))

# Non-significant genes
ns = df_ma[~df_ma["significant"]]
ax.scatter(np.log10(ns["baseMean"] + 1), ns["log2FoldChange"],
           s=3, alpha=0.3, color="grey", label="Not significant", rasterized=True)

# Significant genes
s = df_ma[df_ma["significant"]]
ax.scatter(np.log10(s["baseMean"] + 1), s["log2FoldChange"],
           s=10, alpha=0.7, color="firebrick", label="Significant", rasterized=True)

ax.axhline(0, color="black", lw=0.8, ls="--")
ax.axhline( LFC_THRESHOLD, color="blue", lw=0.6, ls=":")
ax.axhline(-LFC_THRESHOLD, color="blue", lw=0.6, ls=":")
ax.set_xlabel("log10(mean normalised count + 1)")
ax.set_ylabel("log2 Fold Change")
ax.set_title(f"MA plot — {NUMERATOR} vs {DENOMINATOR}")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(OUTDIR / "ma_plot.png", dpi=150)
plt.show()

### 4.2 — Volcano Plot

A volcano plot maps **fold change** against **statistical significance**, making it easy
to spot genes that are both highly significant and strongly changed.

In [ ]:
df_vol = results.dropna(subset=["padj", "log2FoldChange", "pvalue"]).copy()
df_vol["-log10padj"] = -np.log10(df_vol["padj"].clip(lower=1e-300))
df_vol["significant"] = (df_vol["padj"] < FDR_THRESHOLD) & (df_vol["log2FoldChange"].abs() > LFC_THRESHOLD)

# Color categories
def _color(row):
    if not row["significant"]:
        return "grey"
    return "firebrick" if row["log2FoldChange"] > 0 else "steelblue"

df_vol["color"] = df_vol.apply(_color, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))

for color, label in [("grey", "Not significant"),
                     ("firebrick", f"Up in {NUMERATOR}"),
                     ("steelblue", f"Down in {NUMERATOR}")]:
    subset = df_vol[df_vol["color"] == color]
    ax.scatter(subset["log2FoldChange"], subset["-log10padj"],
               s=4, alpha=0.4, color=color, label=f"{label} (n={len(subset):,})",
               rasterized=True)

# Threshold lines
ax.axhline(-np.log10(FDR_THRESHOLD), color="black", lw=0.8, ls="--")
ax.axvline( LFC_THRESHOLD, color="black", lw=0.8, ls=":")
ax.axvline(-LFC_THRESHOLD, color="black", lw=0.8, ls=":")

# Label top genes by adjusted p-value
top_genes = df_vol[df_vol["significant"]].nsmallest(10, "padj")
for gene, row in top_genes.iterrows():
    ax.annotate(gene, (row["log2FoldChange"], row["-log10padj"]),
                fontsize=6, ha="center", va="bottom",
                xytext=(0, 3), textcoords="offset points")

ax.set_xlabel("log2 Fold Change")
ax.set_ylabel("-log10(adjusted p-value)")
ax.set_title(f"Volcano plot — {NUMERATOR} vs {DENOMINATOR}")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.savefig(OUTDIR / "volcano_plot.png", dpi=150)
plt.show()

### 4.3 — Heatmap of Top DE Genes

Visualise the variance-stabilised expression of the top DE genes across samples to
confirm that the groups separate as expected.

PyDESeq2 provides **variance-stabilising transformation (VST)** via
`dds.vst_fit()` / `dds.vst_transform()`, which brings counts onto a roughly
homoscedastic log-scale suitable for heatmaps and PCA.

In [ ]:
# Select top N significant genes (by adjusted p-value)
TOP_N = 50
top_de_genes = (
    results.dropna(subset=["padj"])
    .query("padj < @FDR_THRESHOLD and log2FoldChange.abs() > @LFC_THRESHOLD")
    .nsmallest(TOP_N, "padj")
    .index.tolist()
)

if not top_de_genes:
    print("No significant DE genes found — skipping heatmap.")
else:
    print(f"Plotting heatmap for top {len(top_de_genes)} DE genes.")

    # Use log-normalised counts (log1p of size-factor-normalised counts)
    normed = pd.DataFrame(
        dds.layers["normed_counts"],
        index=dds.obs_names,
        columns=dds.var_names,
    )
    log_normed = np.log1p(normed)

    heatmap_data = log_normed[top_de_genes].T   # genes × samples

    # Annotate columns with condition
    col_colors = metadata["condition"].map(
        {NUMERATOR: "firebrick", DENOMINATOR: "steelblue"}
    )

    g = sns.clustermap(
        heatmap_data,
        col_colors=col_colors,
        z_score=0,              # Z-score per gene (row)
        cmap="RdBu_r",
        figsize=(10, max(6, TOP_N * 0.18)),
        xticklabels=True,
        yticklabels=True,
        linewidths=0,
        dendrogram_ratio=(0.1, 0.05),
        cbar_pos=(0.02, 0.8, 0.03, 0.15),
    )
    g.ax_heatmap.set_xlabel("Samples")
    g.ax_heatmap.set_ylabel("Genes")
    g.fig.suptitle(f"Top {TOP_N} DE genes — {NUMERATOR} vs {DENOMINATOR}",
                   y=1.01, fontsize=12, fontweight="bold")
    g.savefig(OUTDIR / "heatmap_top_de_genes.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 5 — Export Results

Save the full DE results table to a CSV file for downstream analysis.

In [ ]:
out_path = OUTDIR / "de_results.csv"
results.to_csv(out_path)
print(f"DE results saved to: {out_path}")
print(f"Shape: {results.shape}")
display(results.head())

In [ ]:
# Also save the significant subset
sig_path = OUTDIR / "de_results_significant.csv"
sig_results = results.dropna(subset=["padj"]).query(
    "padj < @FDR_THRESHOLD and log2FoldChange.abs() > @LFC_THRESHOLD"
)
sig_results.to_csv(sig_path)
print(f"Significant DE genes saved to: {sig_path}")
print(f"Shape: {sig_results.shape}")

---
## 6 — nf-core `differentialabundance` Pipeline

So far you have performed the analysis manually using Python/PyDESeq2.
In production settings, bioinformaticians often use **standardised, automated pipelines**
that ensure reproducibility and adhere to community best practices.

[**nf-core**](https://nf-co.re/) is a community effort to build high-quality,
peer-reviewed [**Nextflow**](https://www.nextflow.io/) pipelines for common
bioinformatics analyses. The
[`differentialabundance`](https://nf-co.re/differentialabundance) pipeline
wraps multiple DE tools (including DESeq2) into a fully automated, portable workflow.

---

### Step 1 — Install the Required Software

Before running any nf-core pipeline, you need to install **Nextflow** and a
**container engine** (Docker or Singularity/Apptainer).  
Follow the official nf-core installation guide:

| Software | Purpose | Link |
|----------|---------|------|
| **Java ≥ 17** | Required by Nextflow | https://adoptium.net/ |
| **Nextflow** | Workflow manager | https://www.nextflow.io/docs/latest/install.html |
| **Docker** *(recommended)* | Container runtime | https://docs.docker.com/get-docker/ |
| **Singularity / Apptainer** *(HPC)* | Container runtime for HPC | https://apptainer.org/docs/user/latest/quick_start.html |
| **nf-core tools** *(optional)* | Helper CLI for nf-core pipelines | https://nf-co.re/tools |

**Quick install on macOS / Linux:**

```bash
# 1. Install Nextflow
curl -s https://get.nextflow.io | bash
chmod +x nextflow
mv nextflow ~/bin/   # or any directory on your $PATH

# 2. Verify
nextflow -version

# 3. (optional) Install nf-core tools
pip install nf-core
nf-core --version
```

**Useful links:**
- nf-core getting started guide: https://nf-co.re/docs/usage/getting_started/introduction
- nf-core `differentialabundance` pipeline page: https://nf-co.re/differentialabundance
- Pipeline source code: https://github.com/nf-core/differentialabundance

---

### Step 2 — Understand the Required Input Files

Visit the pipeline documentation and study the **Usage** and **Input** sections:

🔗 https://nf-co.re/differentialabundance/latest/docs/usage

The `differentialabundance` pipeline expects three main input files:

| Parameter | File | Description |
|-----------|------|-------------|
| `--input` | `samplesheet.csv` | CSV file describing each sample: `sample`, `fastq_1`, `fastq_2` (or pre-computed matrix paths) |
| `--contrasts` | `contrasts.csv` | CSV file specifying the comparisons to run: `id`, `variable`, `reference`, `target` |
| `--matrix` | `counts.tsv` | Pre-computed count matrix (genes × samples, TSV) — same file you used above |

**Example `samplesheet.csv`** (when using a pre-computed matrix):
```
sample,condition
sample1,control
sample2,control
sample3,treated
sample4,treated
```

**Example `contrasts.csv`**:
```
id,variable,reference,target
treated_vs_control,condition,control,treated
```

> **Task:** Browse the pipeline page and docs, then answer the following questions in the cell below:
> 1. Which DE tools does the pipeline support?
> 2. What does the `--study_type` parameter control?
> 3. What output reports does the pipeline generate?

**Your answers here:**

1. *(DE tools supported: …)*
2. *(study_type: …)*
3. *(output reports: …)*

---

### Step 3 — Run the Pipeline and Compare Outputs

#### 3a — Create the input files

Create the `samplesheet.csv` and `contrasts.csv` files that match your dataset.
You can use the same `data/counts.tsv` and `data/metadata.tsv` files from above.

Run the cells below to generate these files automatically from your metadata.

In [ ]:
nfcore_dir = pathlib.Path("nfcore_run")
nfcore_dir.mkdir(exist_ok=True)

# ── samplesheet.csv ──────────────────────────────────────────────────────────
# The pipeline expects a column named 'sample'; rename the index if needed.
samplesheet = metadata.copy().reset_index()
samplesheet.columns = ["sample"] + list(samplesheet.columns[1:])   # first column → "sample"

samplesheet_path = nfcore_dir / "samplesheet.csv"
samplesheet.to_csv(samplesheet_path, index=False)
print(f"Samplesheet saved to: {samplesheet_path}")
display(samplesheet)

# ── contrasts.csv ────────────────────────────────────────────────────────────
contrasts = pd.DataFrame([
    {
        "id": f"{NUMERATOR}_vs_{DENOMINATOR}",
        "variable": "condition",
        "reference": DENOMINATOR,
        "target": NUMERATOR,
    }
])

contrasts_path = nfcore_dir / "contrasts.csv"
contrasts.to_csv(contrasts_path, index=False)
print(f"\nContrasts saved to: {contrasts_path}")
display(contrasts)

#### 3b — Run the pipeline

Open a terminal in this directory and run the pipeline with the following command
(adapt paths and parameters as needed):

```bash
nextflow run nf-core/differentialabundance \
  -profile docker \
  --study_type rnaseq \
  --matrix data/counts.tsv \
  --input nfcore_run/samplesheet.csv \
  --contrasts nfcore_run/contrasts.csv \
  --outdir nfcore_run/results \
  -resume
```

> **Notes:**
> - Use `-profile singularity` instead of `-profile docker` if you are on an HPC cluster.
> - Add `-r 1.5.0` (or the latest version shown on https://nf-co.re/differentialabundance/releases) to pin a specific release.
> - The `-resume` flag lets you restart from where a previous run stopped.
> - Inspect the pipeline parameters with: `nextflow run nf-core/differentialabundance --help`

#### 3c — Explore the pipeline output

After the pipeline finishes, explore the `nfcore_run/results/` directory.  
Key outputs to look for:

| Output | Location | Description |
|--------|----------|-------------|
| DE results table | `deseq2/` | CSV/TSV with log2FC, p-values, etc. |
| MultiQC report | `multiqc/` | HTML summary of the full run |
| PCA / plots | `plots/` | Diagnostic plots (MA, volcano, PCA) |
| R Markdown report | `report/` | Comprehensive HTML report |

#### 3d — Compare pipeline output to your manual analysis

Load the nf-core pipeline DE results and compare them to the PyDESeq2 results you
generated above.

In [ ]:
# ── Load nf-core results ──────────────────────────────────────────────────────
# Update the path below to point to the actual output file from your pipeline run.
# The file is usually named something like:
#   nfcore_run/results/deseq2/<contrast_id>.deseq2.results.tsv

import glob as _glob

nfcore_result_files = list(pathlib.Path("nfcore_run/results").rglob("*.deseq2.results.tsv"))
if not nfcore_result_files:
    nfcore_result_files = list(pathlib.Path("nfcore_run/results").rglob("*.results.tsv"))

if not nfcore_result_files:
    print("No nf-core result files found yet — run the pipeline first.")
else:
    nfcore_de = pd.read_csv(nfcore_result_files[0], sep="\t", index_col=0)
    print(f"Loaded nf-core results: {nfcore_result_files[0]}")
    print(f"Shape: {nfcore_de.shape}")
    display(nfcore_de.head())

In [ ]:
# ── Compare log2 fold changes ─────────────────────────────────────────────────
# Merge on gene name (index)
try:
    common_genes = results.index.intersection(nfcore_de.index)
    print(f"Genes in common: {len(common_genes):,}")

    comparison = pd.DataFrame({
        "pydeseq2_lfc": results.loc[common_genes, "log2FoldChange"],
        "nfcore_lfc":   nfcore_de.loc[common_genes, "log2FoldChange"],
    }).dropna()

    # Correlation
    corr = comparison.corr().iloc[0, 1]
    print(f"Pearson correlation of log2FC: {corr:.4f}")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(comparison["pydeseq2_lfc"], comparison["nfcore_lfc"],
               s=3, alpha=0.3, color="steelblue", rasterized=True)
    lim = max(comparison.abs().max().max() * 1.1, 1)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("PyDESeq2 log2FC")
    ax.set_ylabel("nf-core differentialabundance log2FC")
    ax.set_title(f"log2FC comparison\n(Pearson r = {corr:.4f})")
    plt.tight_layout()
    plt.savefig(OUTDIR / "comparison_lfc.png", dpi=150)
    plt.show()

except NameError:
    print("Run the pipeline first, then reload this cell.")

#### Discussion questions

Answer these questions in the cell below after completing the comparison:

1. How well do the log2 fold changes from PyDESeq2 and the nf-core pipeline agree?
2. Are there any differences in the number of significant genes reported?
3. What are the advantages of using an automated pipeline like nf-core `differentialabundance`
   compared to a manual Python analysis?
4. What are the potential disadvantages or limitations?

**Your answers here:**

1. *(log2FC agreement: …)*
2. *(differences in significant genes: …)*
3. *(advantages of nf-core: …)*
4. *(disadvantages / limitations: …)*